# TODO

## Purpose: 

{TODO}

In [10]:
import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import glob, gzip, pickle, sys

## Literals


In [2]:
rmats_data_path = "/project/PlatigLab/data/collaborators/BWH/1_ENCODE_shRNA_RBP_KD_2024-04-hg38-gencode-v29/"

rmats_file_column_subset = ["chr", "strand", "exonStart_0base", "exonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE", "IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2", "IncLevel1", "IncLevel2"]

cell_lines = ["K562", "HepG2"]

exon_ordering = {
    "+": {
        "1": "upstreamES", 
        "2": "upstreamEE",
        "3": "exonStart_0base", 
        "4": "exonEnd", 
        "5": "downstreamES", 
        "6": "downstreamEE"
    }, 
    "-": {
        "1": "downstreamEE", 
        "2": "downstreamES",
        "3": "exonEnd",
        "4": "exonStart_0base", 
        "5": "upstreamEE", 
        "6": "upstreamES"
    }
}

sample_names = ["KD-1", "KD-2", "CTRL-1", "CTRL-2"]

## Defining Important Variables

* RBPs
* KD-to-Control Accession IDs
* Expression Values

#### Get the RBPs per Cell Line that we are going to look at

In [3]:
# key is cell line and value is list of rbps
selected_rbps ={}

for cell_line in cell_lines: 
    file = glob.glob("../../5_assign_eCLIP_to_splice_junctions/output/bedtools_input/*{}*_sorted.bed".format(cell_line))
    assert len(file)==1
    
    file
    
    tmp_df = pd.read_csv(file[0], sep="\t", header=None) 

    selected_rbps[cell_line] = sorted(tmp_df[3].str.split("_").str[0].unique().tolist())

['../../5_assign_eCLIP_to_splice_junctions/output/bedtools_input/K562_all_peaks_sorted.bed']

['../../5_assign_eCLIP_to_splice_junctions/output/bedtools_input/HepG2_all_peaks_sorted.bed']

#### Get the Control Accession values per RBP KD

In [4]:
control_associations = {}

for cell_line in cell_lines: 
    
    file = glob.glob("../../3_get_expression_shrna_BAMs/output/2_final_raw_counts_matrices/{}*associations*".format(cell_line))
    assert len(file)==1
    
    file
    
    tmp_df = pd.read_csv(file[0], sep="\t")
        
    control_associations[cell_line] = tmp_df.set_index("RBP KD").to_dict()["Control Accession"]

['../../3_get_expression_shrna_BAMs/output/2_final_raw_counts_matrices/K562_RBP_KD_control_associations.tsv']

['../../3_get_expression_shrna_BAMs/output/2_final_raw_counts_matrices/HepG2_RBP_KD_control_associations.tsv']

#### Get gene expression

## Load and Subset `rMATS Skipped Exon` Files

In [6]:
# dictionary with key as cell line and value is all the rMATS 
# files for that cell line concatted after subsetting for relevant columns 
rmats_cell_line_concat = {}

# for each cell line 
for cell_line in cell_lines: 
    cell_line 
    
    tmp_cell_line_rmats_df = []
    
    # for every SE file 
    for file in sorted(
        glob.glob("{}/*{}*/SE.*".format(rmats_data_path, cell_line), recursive=True)
    ): 

        # get rbp from the file path
        rbp = file.split("/")[-2].split("-")[0]
        # paranoia check: the cell line should be the same as what's labelled on the folder 
        assert file.split("/")[-2].split("-")[2] == cell_line
        
        # select only for RBPs we are looking at and for polyA mRNA samples
        if "-Transfection-" not in file and rbp in selected_rbps[cell_line]: 

            # read rMATS file and subset for relevant columns
            tmp_df = pd.read_csv(file, sep="\t")[rmats_file_column_subset]
            # add RBP KD column 
            tmp_df["RBP_KD_Target"] = rbp
            
            # add df to list of dfs to be concatted 
            tmp_cell_line_rmats_df.append(tmp_df)
    
    
    tmp_cell_line_rmats_df = pd.concat(tmp_cell_line_rmats_df)
    tmp_cell_line_rmats_df.index.size 
    tmp_cell_line_rmats_df.head()
    
    # set dictionary key as cell line and value as concatted dfs 
    rmats_cell_line_concat[cell_line] = tmp_cell_line_rmats_df

'K562'

36525

,chr,strand,exonStart_0base,exonEnd,upstreamES,upstreamEE,downstreamES,downstreamEE,IJC_SAMPLE_1,SJC_SAMPLE_1,IJC_SAMPLE_2,SJC_SAMPLE_2,IncLevel1,IncLevel2,RBP_KD_Target
0,chrX,+,156022698,156022834,156022313,156022459,156023011,156023209,"31,41","1,1","17,15","3,0","0.939,0.953","0.739,1.0",RBFOX2
1,chrX,-,155492356,155492495,155490114,155491666,155506897,155507134,"16,8","1,1","16,12","0,1","0.889,0.8","1.0,0.857",RBFOX2
2,chrX,-,155524455,155524632,155513985,155514265,155545095,155545277,"74,42","1,0","58,50","0,2","0.974,1.0","1.0,0.926",RBFOX2
3,chrX,+,155072326,155072343,155071419,155071650,155077169,155077283,"4,3","0,1","8,4","1,0","1.0,0.719","0.872,1.0",RBFOX2
4,chrX,+,155073373,155073431,155072326,155072343,155077169,155077283,"10,11","0,1","13,4","0,0","1.0,0.874","1.0,1.0",RBFOX2


'HepG2'

32945

,chr,strand,exonStart_0base,exonEnd,upstreamES,upstreamEE,downstreamES,downstreamEE,IJC_SAMPLE_1,SJC_SAMPLE_1,IJC_SAMPLE_2,SJC_SAMPLE_2,IncLevel1,IncLevel2,RBP_KD_Target
0,chrY,+,14719458,14719518,14622008,14622591,14723116,14723269,"24,8","9,6","30,25","8,1","0.624,0.454","0.7,0.94",RBFOX2
1,chrY,+,14748618,14748729,14723116,14723269,14824187,14824373,"0,3","6,10","0,0","10,3","0.0,0.13","0.0,0.0",RBFOX2
2,chrY,+,12909359,12909407,12907536,12907594,12911838,12911968,"239,342","2,2","229,297","0,0","0.988,0.991","1.0,1.0",RBFOX2
3,chrY,+,12911838,12911968,12909359,12909407,12912726,12912882,"798,959","8,1","597,709","1,0","0.98,0.998","0.997,1.0",RBFOX2
4,chrY,+,12912962,12913062,12912726,12912882,12913717,12913853,"1807,1921","0,2","1191,1270","0,0","1.0,0.998","1.0,1.0",RBFOX2


## Separate `PSI` and `Total Counts` Into Separate Columns

In [7]:
for cell_line in rmats_cell_line_concat: 
        
    tmp_df = rmats_cell_line_concat[cell_line]

    tmp_df["PSI_KD-1"] = tmp_df["IncLevel1"].str.split(",").str[0]
    tmp_df["PSI_KD-2"] = tmp_df["IncLevel1"].str.split(",").str[1]

    tmp_df["PSI_CTRL-1"] = tmp_df["IncLevel2"].str.split(",").str[0]
    tmp_df["PSI_CTRL-2"] = tmp_df["IncLevel2"].str.split(",").str[1]

    tmp_df = tmp_df.drop(columns = ["IncLevel1", "IncLevel2"])

    tmp_df["Counts_KD-1"] = (tmp_df["IJC_SAMPLE_1"].str.split(",").str[0]).astype("int64") + (tmp_df["SJC_SAMPLE_1"].str.split(",").str[0]).astype("int64")
    tmp_df["Counts_KD-2"] = (tmp_df["IJC_SAMPLE_1"].str.split(",").str[1]).astype("int64") + (tmp_df["SJC_SAMPLE_1"].str.split(",").str[1]).astype("int64")
    tmp_df["Counts_CTRL-1"] = (tmp_df["IJC_SAMPLE_2"].str.split(",").str[0]).astype("int64") + (tmp_df["SJC_SAMPLE_2"].str.split(",").str[0]).astype("int64")
    tmp_df["Counts_CTRL-2"] = (tmp_df["IJC_SAMPLE_2"].str.split(",").str[1]).astype("int64") + (tmp_df["SJC_SAMPLE_2"].str.split(",").str[1]).astype("int64")
    
    tmp_df.head()
    
    rmats_cell_line_concat[cell_line] = tmp_df.to_dict(orient="records")
    

,chr,strand,exonStart_0base,exonEnd,upstreamES,upstreamEE,downstreamES,downstreamEE,IJC_SAMPLE_1,SJC_SAMPLE_1,IJC_SAMPLE_2,SJC_SAMPLE_2,RBP_KD_Target,PSI_KD-1,PSI_KD-2,PSI_CTRL-1,PSI_CTRL-2,Counts_KD-1,Counts_KD-2,Counts_CTRL-1,Counts_CTRL-2
0,chrX,+,156022698,156022834,156022313,156022459,156023011,156023209,"31,41","1,1","17,15","3,0",RBFOX2,0.939,0.953,0.739,1.0,32,42,20,15
1,chrX,-,155492356,155492495,155490114,155491666,155506897,155507134,"16,8","1,1","16,12","0,1",RBFOX2,0.889,0.8,1.0,0.857,17,9,16,13
2,chrX,-,155524455,155524632,155513985,155514265,155545095,155545277,"74,42","1,0","58,50","0,2",RBFOX2,0.974,1.0,1.0,0.926,75,42,58,52
3,chrX,+,155072326,155072343,155071419,155071650,155077169,155077283,"4,3","0,1","8,4","1,0",RBFOX2,1.0,0.719,0.872,1.0,4,4,9,4
4,chrX,+,155073373,155073431,155072326,155072343,155077169,155077283,"10,11","0,1","13,4","0,0",RBFOX2,1.0,0.874,1.0,1.0,10,12,13,4


,chr,strand,exonStart_0base,exonEnd,upstreamES,upstreamEE,downstreamES,downstreamEE,IJC_SAMPLE_1,SJC_SAMPLE_1,IJC_SAMPLE_2,SJC_SAMPLE_2,RBP_KD_Target,PSI_KD-1,PSI_KD-2,PSI_CTRL-1,PSI_CTRL-2,Counts_KD-1,Counts_KD-2,Counts_CTRL-1,Counts_CTRL-2
0,chrY,+,14719458,14719518,14622008,14622591,14723116,14723269,"24,8","9,6","30,25","8,1",RBFOX2,0.624,0.454,0.7,0.94,33,14,38,26
1,chrY,+,14748618,14748729,14723116,14723269,14824187,14824373,"0,3","6,10","0,0","10,3",RBFOX2,0.0,0.13,0.0,0.0,6,13,10,3
2,chrY,+,12909359,12909407,12907536,12907594,12911838,12911968,"239,342","2,2","229,297","0,0",RBFOX2,0.988,0.991,1.0,1.0,241,344,229,297
3,chrY,+,12911838,12911968,12909359,12909407,12912726,12912882,"798,959","8,1","597,709","1,0",RBFOX2,0.98,0.998,0.997,1.0,806,960,598,709
4,chrY,+,12912962,12913062,12912726,12912882,12913717,12913853,"1807,1921","0,2","1191,1270","0,0",RBFOX2,1.0,0.998,1.0,1.0,1807,1923,1191,1270


## Load `Splice Junction to # RBP Peaks` Data

In [8]:
junction_to_num_peaks = {}

with gzip.GzipFile("../../5_assign_eCLIP_to_splice_junctions/output/splice_junction_rbp_num_peaks/all_RBP_peaks_num_per_splice_junction.pkl.gz", 'rb') as in_file: 
    junction_to_num_peaks = pickle.load(in_file)


## Create Input Data for ML Model by Starting w/ Creating # Peaks per RBP per Sample

#### Pseudocode of Algorithm

In [17]:
# key is cell line and value is a sub-dict where key is distance threshold 
# and value is dataframe represented as dictionary
cell_line_threshold_df = {}

for cell_line in junction_to_num_peaks: 
    
    cell_line_rbps = selected_rbps[cell_line]
    cell_line_controls = control_associations[cell_line]
                                    
    cell_line_threshold_df[cell_line] = {}
    
    for threshold in junction_to_num_peaks[cell_line]:
        
        "{} {}".format(cell_line, threshold)
        
        tmp_junction_to_num_peaks = junction_to_num_peaks[cell_line][threshold]
                
        # dict corresponds to creating a single cell-line-and-threshold specific dataset 
        # where the key is the unique_id described above in the pseudocode and value is 
        # each feature per skipped exon per sample
        ML_input_data = {}
        
        events_encountered = set()
            
        for row in rmats_cell_line_concat[cell_line]: 
            position_definition = exon_ordering[row["strand"]]
            
            for sample in sample_names: 
                
                if row["PSI_"+sample] != "NA": 
                    
                    unique_id = ""
                    
                    unique_id = "_".join(
                        [str(row[coord_column]) for coord_column in rmats_file_column_subset[0:7]]
                    )
                    
                    if "KD" in sample: 
                        kd_ctrl_string = row["RBP_KD_Target"] 
                    elif "CTRL" in sample: 
                        kd_ctrl_string = cell_line_controls[row["RBP_KD_Target"]]
                    
                    unique_id = "_".join(
                        [unique_id, kd_ctrl_string, sample]
                    )
                                        
                    if unique_id not in events_encountered: 
                        
                        events_encountered.add(unique_id)
                        
                        ML_input_data[unique_id] = {}
                        
                        for position in position_definition: 
                            
                            splice_junction_id = "_".join(
                                [row["chr"], str(row[position_definition[position]]), row["strand"]]
                            )
                            
                            if splice_junction_id in tmp_junction_to_num_peaks: 
                                junction_present=True
                            else: 
                                junction_present=False
                                
                            for rbp in cell_line_rbps: 

                                feature_string = "_".join(
                                    [rbp, position, "binding"]
                                ) 

                                if junction_present: 
                                    ML_input_data[unique_id][feature_string] = tmp_junction_to_num_peaks[splice_junction_id][rbp]
                                    
                                elif not junction_present: 
                                    ML_input_data[unique_id][feature_string] = 0
                        
                        ML_input_data[unique_id]["chr"] = row["chr"]
                                                
                        if "KD" in sample: 
                            ML_input_data[unique_id]["RBP_KD_Target"] = row["RBP_KD_Target"]
                        elif "CTRL" in sample: 
                            ML_input_data[unique_id]["RBP_KD_Target"] = "CTRL"                        
                        
                        if sample=="KD-1": 
                            ML_input_data[unique_id]["Inclusion Counts"] = int(row["IJC_SAMPLE_1"].split(",")[0])
                            ML_input_data[unique_id]["Skipping Counts"] = int(row["SJC_SAMPLE_1"].split(",")[0])
                        
                        elif sample=="KD-2": 
                            ML_input_data[unique_id]["Inclusion Counts"] = int(row["IJC_SAMPLE_1"].split(",")[1])
                            ML_input_data[unique_id]["Skipping Counts"] = int(row["SJC_SAMPLE_1"].split(",")[1])
                            
                        elif sample=="CTRL-1": 
                            ML_input_data[unique_id]["Inclusion Counts"] = int(row["IJC_SAMPLE_2"].split(",")[0])
                            ML_input_data[unique_id]["Skipping Counts"] = int(row["SJC_SAMPLE_2"].split(",")[0])
                            
                        elif sample=="CTRL-2": 
                            ML_input_data[unique_id]["Inclusion Counts"] = int(row["IJC_SAMPLE_2"].split(",")[1])
                            ML_input_data[unique_id]["Skipping Counts"] = int(row["SJC_SAMPLE_2"].split(",")[1])
                        
                        ML_input_data[unique_id]["Total Read Counts"] = row["Counts_" + sample]

                        ML_input_data[unique_id]["Target_PSI"] = row["PSI_"+sample]
        
        
        cell_line_threshold_df[cell_line][threshold] = pd.DataFrame.from_dict(ML_input_data, orient="index")
                        
        

'K562 50'

'K562 100'

'K562 500'

'K562 1000'

'K562 2000'

...

'HepG2 50'

'HepG2 100'

'HepG2 500'

'HepG2 1000'

KeyboardInterrupt: 